In [1]:
import lfox
import lfox.lattice as lat
import lfox.evolution.hmc as lhmc
import jax
import jax.numpy as jnp
import numpy as np

# Imports below require "dev" environment
import matplotlib.pyplot as plt
import lsqfit
import gvar as gv
import tqdm

# Double precision!
jax.config.update("jax_enable_x64", True)
jax.config.update("jax_threefry_partitionable", True)


In [2]:
class ScalarAction(lhmc.Action):        

    @staticmethod
    @jax.jit
    def _S(fields, params):
        phi = fields[0]
        S = phi**2
        
        for ax in range(phi.d):
            S -= 2 * params['kappa'] * phi * phi.nn_field(ax)

        S += params['lambda'] * (phi**2 - 1)**2
        return S
    
    # Exact force function instead of autodiff, for testing purposes
    @staticmethod
    @jax.jit
    def exact_force(fields, params):
        phi = fields[0]

        J = phi.nn_field(axis=0, shift=1) + phi.nn_field(axis=0, shift=-1)
        for ax in range(1,phi.d):
            J += phi.nn_field(axis=ax, shift=1)
            J += phi.nn_field(axis=ax, shift=-1)

        F = -2 * params['kappa'] * J
        F += 2 * phi.F
        F += 4 * params['lambda'] * (phi.F**2 - 1) * phi.F

        return F

In [3]:
def show_live(verbose=False):
    if verbose:
        print("Live details: ")
        print(jax.live_arrays())
        print("------------------------------")
    
    print("Live arrays = ", len(jax.live_arrays()))

In [4]:
lat.SquareLattice(st_dims=((4,4)))
lat.HoneycombLattice(st_dims=((4,4)))

HoneycombLattice(st_dims=(4, 4), _dims=(4, 4, 2))

In [5]:
d = 3
Lat4 = lat.SquareLattice(st_dims=((4,)*d))
phi4 = lat.LatticeField(lattice=Lat4, F=1)
#phi4.F = np.ones_like(phi4.F)
#phi4.set_field(np.ones_like(phi4.F))

show_live()  # Expect 1: {phi4}

S4 = ScalarAction(field_names=['phi'], params={'kappa': 0.18169, 'lambda': 1.3282})

show_live()  # Expect 1: {phi4}


Live arrays =  2
Live arrays =  2


I0000 00:00:1704577435.527711       1 tfrt_cpu_pjrt_client.cc:349] TfrtCpuClient created.


In [6]:
len(phi4.bc)

3

In [7]:
import dataclasses

print(dataclasses.asdict(phi4))

print(phi4.lattice)
lat.LatticeField(**dataclasses.asdict(phi4))

{'lattice': {'st_dims': (4, 4, 4), '_dims': (4, 4, 4)}, 'F': Array([[[1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.]],

       [[1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.]],

       [[1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.]],

       [[1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.]]], dtype=float64), 'bc': Array([1., 1., 1.], dtype=float64), 'indices': ()}
SquareLattice(st_dims=(4, 4, 4), _dims=(4, 4, 4))


AttributeError: 'dict' object has no attribute 'st_dims'

In [8]:
print(phi4.copy())
print(phi4.copy_new_F(1))
p2 = phi4 + phi4
print(p2)
print(phi4.F, p2.F)

LatticeField(
  lattice=SquareLattice(st_dims=(4, 4, 4), _dims=(4, 4, 4)),
  F=f64[4,4,4],
  bc=f64[3],
  indices=()
)
LatticeField(
  lattice=SquareLattice(st_dims=(4, 4, 4), _dims=(4, 4, 4)),
  F=f64[4,4,4],
  bc=f64[3],
  indices=()
)
LatticeField(
  lattice=SquareLattice(st_dims=(4, 4, 4), _dims=(4, 4, 4)),
  F=f64[4,4,4],
  bc=f64[3],
  indices=()
)
[[[1. 1. 1. 1.]
  [1. 1. 1. 1.]
  [1. 1. 1. 1.]
  [1. 1. 1. 1.]]

 [[1. 1. 1. 1.]
  [1. 1. 1. 1.]
  [1. 1. 1. 1.]
  [1. 1. 1. 1.]]

 [[1. 1. 1. 1.]
  [1. 1. 1. 1.]
  [1. 1. 1. 1.]
  [1. 1. 1. 1.]]

 [[1. 1. 1. 1.]
  [1. 1. 1. 1.]
  [1. 1. 1. 1.]
  [1. 1. 1. 1.]]] [[[2. 2. 2. 2.]
  [2. 2. 2. 2.]
  [2. 2. 2. 2.]
  [2. 2. 2. 2.]]

 [[2. 2. 2. 2.]
  [2. 2. 2. 2.]
  [2. 2. 2. 2.]
  [2. 2. 2. 2.]]

 [[2. 2. 2. 2.]
  [2. 2. 2. 2.]
  [2. 2. 2. 2.]
  [2. 2. 2. 2.]]

 [[2. 2. 2. 2.]
  [2. 2. 2. 2.]
  [2. 2. 2. 2.]
  [2. 2. 2. 2.]]]


In [9]:

HMC = lhmc.HMCRewrite(
    action=S4,
    seed=72345,
    fields={'phi': phi4},
    integrator=lhmc.LeapfrogIntegrator(eps=0.1, Nstep=10),
    observables={},
    save_freq=1,
)

HMC.evolve()

OK


In [10]:

HMC = lhmc.HMCEvolver(
    action=S4,
    seed=72345,
    init_fields={'phi': phi4},
    integrator=lhmc.LeapfrogIntegrator(eps=0.1, Nstep=10),
)

show_live()  # Expect 3: {phi4, HMC.rng_key, HMC.pi_fields}

HMC.evolve(warmup=False)

show_live()  # Expect 4: above plus new entry in field_chain

HMC.evolve(warmup=True)

show_live()  # Expect 5: above plus new entry in field_chain

#HMC.evolve(warmup=True)

#show_live()


Live arrays =  8


AttributeError: 'LatticeField' object has no attribute 'd'

In [8]:
jax.live_arrays()

[Array([[[-1.05856642, -0.14057331,  1.0143616 , -0.5392155 ],
         [-0.99988364, -0.19330283,  0.8148847 , -0.26844978],
         [-0.2923622 , -0.76299877,  0.26591322,  1.00225178],
         [-0.28325348, -0.06318021, -0.37045691, -0.25804596]],
 
        [[-1.02501982, -1.26452435, -0.62650538, -1.33208245],
         [-0.40985767, -0.06375308,  0.20814922, -0.97204486],
         [-1.07884784, -1.27570952, -0.41077675, -0.28424382],
         [-1.08361778, -0.92698217, -0.08835638,  0.77264094]],
 
        [[-0.02141358,  0.68343794,  0.28716798, -0.79467947],
         [ 0.90897115,  0.70159878, -0.1183697 ,  0.64069117],
         [ 0.1174347 , -1.10425718, -0.93607623, -0.72753582],
         [ 1.23237295, -1.20665012,  1.15459966,  0.71856502]],
 
        [[-0.83682517,  1.08158739, -0.53488309, -1.0765904 ],
         [ 1.36513197,  1.12062112, -0.3031387 , -0.74039191],
         [-1.15242664,  0.23700415, -0.75455302, -0.842065  ],
         [-0.31719168,  0.39358456, -0.1463635

In [9]:
HMC.__dict__

{'integrator': LeapfrogIntegrator(eps=0.1, Nstep=10),
 'monitor': {'delta_H': [-0.07937245886360289,
   -0.03578238935186495,
   0.046221853735688434],
  'P_acc': [1.0826074737363443, 1.0364302836836004, 0.9548301060650688],
  'accept': [True, True, True]},
 'traj_init': 0,
 'traj_chain': [0, 1, 2, 3],
 'action': ScalarAction(
   field_names=['phi'],
   params={'kappa': 0.18169, 'lambda': 1.3282},
   sub_actions=[]
 ),
 'seed': 72345,
 'rng_key': Array([2513059081, 1870155651], dtype=uint32),
 'fields': {'phi': <lfox.lattice.LatticeField at 0x126b54250>},
 'save_freq': 1,
 'observables': None,
 'field_chain': {'phi': [<lfox.lattice.LatticeField at 0x127ceddd0>,
   <lfox.lattice.LatticeField at 0x126b54250>]},
 'N_fields': 1,
 'delta_P': <function lfox.evolution.hmc.HMCEvolver.delta_mom.<locals>.delta_P(X, P)>,
 'delta_X': <function lfox.evolution.hmc.HMCEvolver.delta_fields.<locals>.delta_X(X, P)>,
 'pi_fields': {'phi': <lfox.lattice.LatticeField at 0x127fdbe50>}}